<a href="https://colab.research.google.com/github/Baskar412/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [7]:
with open('sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.1 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal",
    "Solved 250+ DSA problems on LeetCode",
    "Final-year project: ML model for crop yield prediction"
  ],
  "experience_years": 0.25
}


In [10]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2020},{"degree":"Bachelor of Science in Software Engineering","institution":"Another University","year":2018}],"skills":["Python","Java","C++","JavaScript","React","SQL","AWS","Docker"],"projects":["E-commerce Platform","AI Chatbot","Data Analysis Dashboard"],"experience_years":3.5}


## Day 2 Lab 2B — Errors handled

1. **Markdown fence wrapping** (`\`\`\`json ... \`\`\``)
   The retry prompt asks Gemini to output raw JSON without fences. Triggers on ~5-10% of calls.

2. **Hallucinated phone number when source has none**
   `Optional[str] = None` in Pydantic — model returns `null`, schema validates.

3. **Empty / whitespace-only input**
   Pydantic raises ValidationError with "Field required". Caller catches.

## Sample résumés processed: 3 / 3 successful

In [12]:
from google.colab import files

uploaded = files.upload()  # opens a file picker; choose 10_mcq_questions_with_answers.txt
input_filename = next(iter(uploaded))   # name of the file you just uploaded
print("Uploaded:", input_filename)

Saving 10_mcq_questions_with_answers.txt to 10_mcq_questions_with_answers (1).txt
Uploaded: 10_mcq_questions_with_answers (1).txt


In [13]:
import json
import re
from pathlib import Path

def parse_mcq_file(input_path):
    text = Path(input_path).read_text(encoding="utf-8").strip()
    blocks = re.split(r"\n\s*\n", text)   # split on blank lines
    questions = []

    for block in blocks:
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        if not lines:
            continue

        q_match = re.match(r"^(\d+)\.\s*(.+)$", lines[0])
        if not q_match:
            continue

        options, answer = {}, None
        for line in lines[1:]:
            opt = re.match(r"^([A-D])\.\s*(.+)$", line)
            ans = re.match(r"^Answer:\s*([A-D])\s*$", line, re.IGNORECASE)
            if opt:
                options[opt.group(1)] = opt.group(2).strip()
            elif ans:
                answer = ans.group(1).upper()

        questions.append({
            "question_number": int(q_match.group(1)),
            "question": q_match.group(2).strip(),
            "options": options,
            "answer": answer,
        })
    return questions

output_filename = "mcq_questions.json"
questions = parse_mcq_file(input_filename)

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)

print(f"Converted {len(questions)} questions -> {output_filename}")
questions[:2]   # quick preview of the first two

Converted 10 questions -> mcq_questions.json


[{'question_number': 1,
  'question': 'Which data structure follows the LIFO principle?',
  'options': {'A': 'Queue', 'B': 'Stack', 'C': 'Array', 'D': 'Linked List'},
  'answer': 'B'},
 {'question_number': 2,
  'question': 'Which keyword is used to create a class in Java?',
  'options': {'A': 'object', 'B': 'struct', 'C': 'class', 'D': 'define'},
  'answer': 'C'}]